In [2]:
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
print('Libraries loaded successfully')

Libraries loaded successfully


In [3]:
df = pd.read_csv('../data/synthetic_rwandan.csv')

print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nFirst 5 rows:')
print(df.head())
print('\nData types:')
print(df.dtypes)
print('\nNull values:')
print(df.isnull().sum())
print('\nTarget column (bg_2h_later) statistics:')
print(df['bg_2h_later'].describe())

Shape: (28800, 10)

Columns: ['fasting_bg', 'current_bg', 'bg_2h_later', 'meal_gi', 'meal_calories', 'activity_encoded', 'insulin_dose', 'hour_of_day', 'day', 'patient_id']

First 5 rows:
   fasting_bg  current_bg  bg_2h_later  meal_gi  meal_calories  \
0       164.3       188.7        158.6       45            400   
1       164.3       183.6        128.2       45            620   
2       164.3       172.0        142.5       71            620   
3       170.8       190.1        155.4       71            620   
4       170.8       203.9        172.7       71            400   

   activity_encoded  insulin_dose  hour_of_day  day  patient_id  
0                 0             0            7    0           0  
1                 3             0           13    0           0  
2                 0             0           19    0           0  
3                 0             0            7    1           0  
4                 1             0           13    1           0  

Data types:
fastin

In [4]:
# Features match exactly what Node.js sends to Flask (CLAUDE.md 7.3)
# Create lag features from patient history within each patient
df = df.sort_values(['patient_id', 'day', 'hour_of_day']).reset_index(drop=True)

# Create lag BG readings per patient (simulates last N readings)
df['bg_lag_1'] = df.groupby('patient_id')['current_bg'].shift(1)
df['bg_lag_2'] = df.groupby('patient_id')['current_bg'].shift(2)
df['bg_lag_3'] = df.groupby('patient_id')['current_bg'].shift(3)
df['bg_lag_4'] = df.groupby('patient_id')['current_bg'].shift(4)

# Fill nulls for first readings of each patient with current_bg
df['bg_lag_1'] = df['bg_lag_1'].fillna(df['current_bg'])
df['bg_lag_2'] = df['bg_lag_2'].fillna(df['current_bg'])
df['bg_lag_3'] = df['bg_lag_3'].fillna(df['current_bg'])
df['bg_lag_4'] = df['bg_lag_4'].fillna(df['current_bg'])

# Minutes since meal approximation from hour of day
def estimate_minutes_since_meal(hour):
    if 6 <= hour <= 9:   return 30
    if 12 <= hour <= 14: return 30
    if 18 <= hour <= 21: return 30
    return 120

df['minutes_since_meal'] = df['hour_of_day'].apply(estimate_minutes_since_meal)

# Define feature columns — must match Flask endpoint exactly
FEATURE_COLS = [
    'fasting_bg',        # bg_1 (oldest)
    'bg_lag_4',          # bg_2
    'bg_lag_3',          # bg_3
    'bg_lag_2',          # bg_4
    'current_bg',        # bg_5 (most recent)
    'meal_gi',
    'meal_calories',
    'activity_encoded',
    'insulin_dose',
    'hour_of_day',
    'minutes_since_meal',
]

TARGET_COL = 'bg_2h_later'

X = df[FEATURE_COLS]
y = df[TARGET_COL]

print('Feature matrix shape:', X.shape)
print('Target shape:', y.shape)
print('\nFeature columns:', FEATURE_COLS)
print('\nFeature statistics:')
print(X.describe())

Feature matrix shape: (28800, 11)
Target shape: (28800,)

Feature columns: ['fasting_bg', 'bg_lag_4', 'bg_lag_3', 'bg_lag_2', 'current_bg', 'meal_gi', 'meal_calories', 'activity_encoded', 'insulin_dose', 'hour_of_day', 'minutes_since_meal']

Feature statistics:
         fasting_bg      bg_lag_4      bg_lag_3      bg_lag_2    current_bg  \
count  28800.000000  28800.000000  28800.000000  28800.000000  28800.000000   
mean     166.948333    192.945906    193.024160    193.102028    193.267788   
std       60.260192     61.255193     61.316744     61.384715     61.520426   
min       80.000000     80.100000     80.100000     80.100000     80.100000   
25%      117.300000    143.600000    143.600000    143.600000    143.600000   
50%      160.500000    186.300000    186.400000    186.400000    186.600000   
75%      207.400000    233.700000    233.800000    234.000000    234.325000   
max      350.000000    400.000000    400.000000    400.000000    400.000000   

            meal_gi  meal_

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples:     {X_test.shape[0]}')
print(f'Features:         {X_train.shape[1]}')

Training samples: 23040
Test samples:     5760
Features:         11


In [6]:
print('Training Random Forest Regressor...')

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)
print('Training complete')

Training Random Forest Regressor...
Training complete


In [7]:
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print('=' * 40)
print('MODEL EVALUATION RESULTS')
print('=' * 40)
print(f'RMSE: {rmse:.2f} mg/dL  (target: < 20)')
print(f'MAE:  {mae:.2f} mg/dL')
print(f'R²:   {r2:.4f}')
print('=' * 40)

if rmse < 20:
    print('✅ RMSE target met — model is production ready')
elif rmse < 30:
    print('⚠️  RMSE acceptable — model will work but could be improved')
else:
    print('❌ RMSE too high — model needs tuning before deployment')

# Cross validation
cv_scores = cross_val_score(
    model, X, y,
    scoring='neg_root_mean_squared_error',
    cv=5, n_jobs=-1
)
print(f'\nCross-validation RMSE: {-cv_scores.mean():.2f} ± {cv_scores.std():.2f} mg/dL')

MODEL EVALUATION RESULTS
RMSE: 10.16 mg/dL  (target: < 20)
MAE:  8.06 mg/dL
R²:   0.9728
✅ RMSE target met — model is production ready

Cross-validation RMSE: 10.18 ± 0.07 mg/dL


In [8]:
importances = pd.Series(
    model.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=False)

print('Feature Importances:')
print(importances.round(4))

Feature Importances:
current_bg            0.9632
activity_encoded      0.0224
bg_lag_2              0.0032
bg_lag_4              0.0032
bg_lag_3              0.0032
fasting_bg            0.0030
hour_of_day           0.0007
meal_calories         0.0007
meal_gi               0.0004
insulin_dose          0.0000
minutes_since_meal    0.0000
dtype: float64


In [9]:
sample_indices = np.random.choice(len(y_test), 10, replace=False)
sample_actual = y_test.iloc[sample_indices].values
sample_pred = y_pred[sample_indices]

print('Sample predictions vs actual values:')
print(f'{"Actual":>10} {"Predicted":>12} {"Error":>10}')
print('-' * 35)
for actual, pred in zip(sample_actual, sample_pred):
    error = abs(actual - pred)
    print(f'{actual:>10.1f} {pred:>12.1f} {error:>10.1f}')

Sample predictions vs actual values:
    Actual    Predicted      Error
-----------------------------------
     263.9        255.6        8.3
     179.6        168.4       11.2
     186.5        189.8        3.3
     251.1        247.4        3.7
     129.2        126.0        3.2
     206.2        183.5       22.7
     170.4        181.7       11.3
     114.8        122.2        7.4
     260.6        260.0        0.6
     265.3        279.3       14.0


In [10]:
os.makedirs('../models', exist_ok=True)
model_path = '../models/glucose_model.pkl'
joblib.dump(model, model_path)

file_size = os.path.getsize(model_path) / (1024 * 1024)
print(f'Model saved to: {model_path}')
print(f'File size: {file_size:.1f} MB')

# Save feature column names so Flask can validate input order
feature_config = {
    'feature_cols': FEATURE_COLS,
    'target_col': TARGET_COL,
    'model_version': '1.0.0',
    'rmse': round(rmse, 2),
    'mae': round(mae, 2),
    'r2': round(r2, 4),
    'training_samples': X_train.shape[0],
}
joblib.dump(feature_config, '../models/glucose_model_config.pkl')
print('Feature config saved to: ../models/glucose_model_config.pkl')

Model saved to: ../models/glucose_model.pkl
File size: 50.6 MB
Feature config saved to: ../models/glucose_model_config.pkl


In [ ]:
loaded_model = joblib.load(model_path)

import pandas as pd

# Simulate exactly what Flask receives from Node.js
test_input = pd.DataFrame([[
    145.0, 132.0, 178.0, 145.0, 165.0,
    71, 620, 0, 0, 13, 30,
]], columns=[
    'fasting_bg', 'bg_lag_4', 'bg_lag_3', 'bg_lag_2', 'current_bg',
    'meal_gi', 'meal_calories', 'activity_encoded', 'insulin_dose',
    'hour_of_day', 'minutes_since_meal',
])

test_prediction = loaded_model.predict(test_input)[0]
print(f'Test prediction for Ugali meal at 1pm:')
print(f'  Current BG: 165 mg/dL')
print(f'  Predicted BG in 2 hours: {test_prediction:.1f} mg/dL')
print(f'  Model loaded and working correctly ✅')

print('\n' + '=' * 40)
print('STEP 3.3 COMPLETE')
print(f'Model file: ../models/glucose_model.pkl')
print(f'RMSE: {rmse:.2f} mg/dL')
print('Run Step 3.4 next to train the risk classifier')
print('=' * 40)

Test prediction for Ugali meal at 1pm:
  Current BG: 165 mg/dL
  Predicted BG in 2 hours: 144.7 mg/dL
  Model loaded and working correctly...

STEP 3.3 COMPLETE
Model file: ../models/glucose_model.pkl
RMSE: 10.16 mg/dL
Run Step 3.4 next to train the risk classifier


C:\Users\USER\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
